# CartPole + GradientMonitoredNet: DQN, Discrete SAC, PPO

Standalone pipeline for testing gradient monitoring on CartPole-v1.

Each section:
1. Creates `GradientMonitoredNet` / `GradientMonitoredBaseNet` with hooks
2. Builds the tianshou algorithm (DQN / DiscreteSAC / PPO)
3. Trains for several epochs with `grad_verbose=True`
4. Gradient stats are printed every `grad_log_interval` backward steps

**Log format:**
```
[GradMonitor] GradientMonitoredNet/q_net#1 | step 500
  model.0.weight     | norm: avg=0.05 max=0.11 | val: min=-0.03 max=0.03 | mean_abs=0.001
```
- `norm avg/max` — L2-norm of gradient tensor (averaged / max over interval)
- `val min/max` — element-wise min/max gradient value
- `mean_abs` — mean |grad|, proxy for average gradient magnitude
- `[EXPLODING]` — max_norm > 100
- `[VANISHING]` — mean_abs < 1e-7

In [1]:
import gymnasium as gym
import torch
import torch.nn as nn
import numpy as np
import tianshou as ts
import tianshou.algorithm.optim as opt

from tianshou.algorithm.modelfree.dqn import DQN, DiscreteQLearningPolicy
from tianshou.algorithm.modelfree.discrete_sac import DiscreteSAC, DiscreteSACPolicy
from tianshou.algorithm.modelfree.ppo import PPO
from tianshou.algorithm.modelfree.reinforce import ProbabilisticActorPolicy
from tianshou.algorithm.modelfree.sac import AutoAlpha
from tianshou.utils.net.discrete import DiscreteActor, DiscreteCritic
from tianshou.data import VectorReplayBuffer, Collector
from tianshou.trainer import (
    OffPolicyTrainer, OffPolicyTrainerParams,
    OnPolicyTrainer, OnPolicyTrainerParams,
)
from tianshou.utils.net.common import Net

from hpo_rl.nets.gradient_monitor import (
    GradientMonitoredNet,
    GradientMonitoredBaseNet,
    GradientMonitorMixin,
)

print(f"PyTorch: {torch.__version__}")
print(f"Tianshou: {ts.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

PyTorch: 2.9.1+cpu
Tianshou: 2.0.0
Device: cpu


## Verify GradientMonitoredNet architecture

In [2]:
# CartPole: state_shape=4, action_shape=2 (Discrete)
GradientMonitorMixin.reset_instance_counter()

net_test = GradientMonitoredNet(
    state_shape=4, action_shape=2,
    hidden_sizes=[128, 128],
    grad_log_interval=5, grad_verbose=False,
)
print("Architecture:")
print(net_test.model)
print(f"\noutput_dim = {net_test.output_dim}")
print(f"_has_output_head = {net_test._has_output_head}")
print(f"Name in logs: {net_test._gm_name}")
print(f"Tracked params: {net_test._num_tracked}")

# Quick forward pass
obs = torch.randn(8, 4)
out, _ = net_test(obs)
print(f"\nForward: input={obs.shape} -> output={out.shape}")

net_test.remove_hooks()
del net_test

Architecture:
Sequential(
  (0): Linear(in_features=4, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=128, bias=True)
  (3): ReLU()
  (4): Linear(in_features=128, out_features=2, bias=True)
)

output_dim = 2
_has_output_head = True
Name in logs: GradientMonitoredNet/q_net#1
Tracked params: 6

Forward: input=torch.Size([8, 4]) -> output=torch.Size([8, 2])


---
## 1. DQN on CartPole-v1

Value-based method. One `GradientMonitoredNet` acts as Q-network.  
Tianshou internally creates a target net via `deepcopy` — our `__deepcopy__` override ensures it gets **no hooks**.

In [3]:
# ── DQN on CartPole ──────────────────────────────────────────────
GradientMonitorMixin.reset_instance_counter()

N_TRAIN_ENVS = 8
N_TEST_ENVS  = 4
GRAD_LOG_INTERVAL = 200
GRAD_VERBOSE = True

# 1. Environments
train_envs = ts.env.DummyVectorEnv([lambda: gym.make("CartPole-v1") for _ in range(N_TRAIN_ENVS)])
test_envs  = ts.env.DummyVectorEnv([lambda: gym.make("CartPole-v1") for _ in range(N_TEST_ENVS)])

# 2. Q-Network
q_net = GradientMonitoredNet(
    state_shape=4,
    action_shape=2,
    hidden_sizes=[128, 128],
    grad_log_interval=GRAD_LOG_INTERVAL,
    grad_verbose=GRAD_VERBOSE,
)
print(f"Q-net name: {q_net._gm_name}")

# 3. Policy + Algorithm
dqn_optim = opt.TorchOptimizerFactory(optim_class=torch.optim.Adam, lr=1e-3)

dqn_policy = DiscreteQLearningPolicy(
    model=q_net,
    action_space=gym.make("CartPole-v1").action_space,
    eps_training=0.1,
    eps_inference=0.0,
)

dqn_algo = DQN(
    policy=dqn_policy,
    optim=dqn_optim,
    gamma=0.99,
    target_update_freq=200,
)

# 4. Collectors + Buffer
dqn_buffer = VectorReplayBuffer(total_size=20000, buffer_num=N_TRAIN_ENVS)
dqn_train_collector = Collector(dqn_algo, train_envs, dqn_buffer)
dqn_test_collector  = Collector(dqn_algo, test_envs)

# 5. Train
dqn_params = OffPolicyTrainerParams(
    max_epochs=10,
    epoch_num_steps=1000,
    training_collector=dqn_train_collector,
    test_collector=dqn_test_collector,
    collection_step_num_env_steps=200,
    update_step_num_gradient_steps_per_sample=0.5,
    test_step_num_episodes=N_TEST_ENVS,
    batch_size=64,
    show_progress=True,
)

dqn_result = OffPolicyTrainer(algorithm=dqn_algo, params=dqn_params).run()
print(f"\nDQN best reward: {dqn_result.best_reward:.1f}")
print(f"DQN total backward steps: {q_net._grad_step}")

# Cleanup
q_net.remove_hooks()
train_envs.close()
test_envs.close()

Q-net name: GradientMonitoredNet/q_net#1
Initial test step: test_reward: 9.500000 ± 0.866025, best_reward: 9.500000 ± 0.866025 in #0


[GradMonitor] GradientMonitoredNet/q_net#1 | step 200
  model.0.bias                                       | norm: avg=0.021237 max=0.234946 | val: min=-0.086862 max=0.048198 | mean_abs=0.001132
  model.0.weight                                     | norm: avg=0.033047 max=0.356783 | val: min=-0.078461 max=0.094739 | mean_abs=0.000608
  model.2.bias                                       | norm: avg=0.032632 max=0.737678 | val: min=-0.161970 max=0.163027 | mean_abs=0.001873
  model.2.weight                                     | norm: avg=0.132340 max=2.609535 | val: min=-0.181051 max=0.182232 | mean_abs=0.000390
  model.4.bias                                       | norm: avg=0.083970 max=1.993477 | val: min=-1.993477 max=0.211426 | mean_abs=0.041985
  model.4.weight                                     | norm: avg=0.211006 max=3.377105 | val: min=-1.012584 max=0.416102 | mean_abs=0.005485


[GradMonitor] GradientMonitoredNet/q_net#1 | step 400
  model.0.bias                                       | norm: avg=0.096146 max=0.762436 | val: min=-0.190881 max=0.143890 | mean_abs=0.005259
  model.0.weight                                     | norm: avg=0.195988 max=1.695562 | val: min=-0.335376 max=0.225427 | mean_abs=0.003938
  model.2.bias                                       | norm: avg=0.055089 max=0.694486 | val: min=-0.172233 max=0.090536 | mean_abs=0.002989
  model.2.weight                                     | norm: avg=0.248379 max=2.476593 | val: min=-0.200340 max=0.182730 | mean_abs=0.000684
  model.4.bias                                       | norm: avg=0.120977 max=1.783420 | val: min=-1.783420 max=0.830063 | mean_abs=0.060488
  model.4.weight                                     | norm: avg=0.526093 max=5.603453 | val: min=-1.445414 max=1.183897 | mean_abs=0.014080


Epoch #1: 100%|##########| 1000/1000 [00:01<00:00, 704.97it/s, env_episode=102, env_step=1000, len=9, n_ep=21, n_st=200, rew=9.67, update_step=5]

Epoch #1: test_reward: 9.500000 ± 0.500000, best_reward: 9.500000 ± 0.866025 in #0


...

Epoch #9: test_reward: 9.500000 ± 0.500000, best_reward: 10.000000 ± 0.707107 in #2


[GradMonitor] GradientMonitoredNet/q_net#1 | step 4600
  model.0.bias                                       | norm: avg=0.187246 max=0.550766 | val: min=-0.163211 max=0.157686 | mean_abs=0.010000
  model.0.weight                                     | norm: avg=0.326124 max=1.059993 | val: min=-0.307790 max=0.302437 | mean_abs=0.006095
  model.2.bias                                       | norm: avg=0.040809 max=0.119427 | val: min=-0.030893 max=0.030736 | mean_abs=0.002170
  model.2.weight                                     | norm: avg=0.160066 max=0.454700 | val: min=-0.040672 max=0.044070 | mean_abs=0.000446
  model.4.bias                                       | norm: avg=0.048495 max=0.147797 | val: min=-0.144102 max=0.147797 | mean_abs=0.024248
  model.4.weight                                     | norm: avg=0.325551 max=1.055550 | val: min=-0.202494 max=0.190481 | mean_abs=0.008747


[GradMonitor] GradientMonitoredNet/q_net#1 | step 4800
  model.0.bias                                       | norm: avg=0.221201 max=0.641300 | val: min=-0.229141 max=0.185815 | mean_abs=0.011910
  model.0.weight                                     | norm: avg=0.378248 max=1.565592 | val: min=-0.382851 max=0.477165 | mean_abs=0.007087
  model.2.bias                                       | norm: avg=0.048640 max=0.147469 | val: min=-0.033969 max=0.035703 | mean_abs=0.002605
  model.2.weight                                     | norm: avg=0.183823 max=0.696255 | val: min=-0.048446 max=0.060259 | mean_abs=0.000510
  model.4.bias                                       | norm: avg=0.059625 max=0.187519 | val: min=-0.182239 max=0.187519 | mean_abs=0.029813
  model.4.weight                                     | norm: avg=0.373308 max=0.969216 | val: min=-0.249734 max=0.280567 | mean_abs=0.010075


Epoch #10: 100%|##########| 1000/1000 [00:01<00:00, 701.07it/s, env_episode=1066, env_step=10000, len=9, n_ep=21, n_st=200, rew=9.33, update_step=50]

[GradMonitor] GradientMonitoredNet/q_net#1 | step 5000
  model.0.bias                                       | norm: avg=0.173026 max=0.541328 | val: min=-0.166456 max=0.155905 | mean_abs=0.009279
  model.0.weight                                     | norm: avg=0.303062 max=0.956510 | val: min=-0.266756 max=0.284382 | mean_abs=0.005670
  model.2.bias                                       | norm: avg=0.037697 max=0.125058 | val: min=-0.028353 max=0.032737 | mean_abs=0.002004
  model.2.weight                                     | norm: avg=0.147592 max=0.463196 | val: min=-0.036163 max=0.042607 | mean_abs=0.000407
  model.4.bias                                       | norm: avg=0.045496 max=0.159966 | val: min=-0.153838 max=0.159966 | mean_abs=0.022748
  model.4.weight                                     | norm: avg=0.295454 max=0.931314 | val: min=-0.180699 max=0.192049 | mean_abs=0.007932
Epoch #10: test_reward: 9.250000 ± 0.829156, best_reward: 10.000000 ± 0.707107 in #2

DQN best rewa

## 2. PPO on CartPole-v1

On-policy actor-critic. Two `GradientMonitoredBaseNet` instances:
- **actor#1** — actor's preprocess net (→ DiscreteActor head)
- **critic#2** — critic's preprocess net (→ DiscreteCritic head)

**Dual monitoring:**
- `register_post_accumulate_grad_hook` → **pre-clip** gradient stats (fires after `backward()`)
- `OptimizerStepMonitor` → **post-clip** gradient stats (fires after `clip_grad_norm_`)

This shows exactly how much `max_grad_norm=0.5` reduces critic gradient norms.

In [5]:
# ── PPO on CartPole ──────────────────────────────────────────────
import time
GradientMonitorMixin.reset_instance_counter()

# 1. Environments
train_envs = ts.env.DummyVectorEnv([lambda: gym.make("CartPole-v1") for _ in range(N_TRAIN_ENVS)])
test_envs  = ts.env.DummyVectorEnv([lambda: gym.make("CartPole-v1") for _ in range(N_TEST_ENVS)])

# 2. Actor + Critic preprocess nets
net_actor = GradientMonitoredBaseNet(
    state_shape=4,
    hidden_sizes=[128, 128],
    grad_log_interval=GRAD_LOG_INTERVAL,
    grad_verbose=GRAD_VERBOSE,
    grad_monitor_name="actor",
)

net_critic = GradientMonitoredBaseNet(
    state_shape=4,
    hidden_sizes=[128, 128],
    grad_log_interval=GRAD_LOG_INTERVAL,
    grad_verbose=GRAD_VERBOSE,
    grad_monitor_name="critic",
)

print(f"Actor net:  {net_actor._gm_name}")
print(f"Critic net: {net_critic._gm_name}")

# 3. Actor + Critic heads
actor  = DiscreteActor(preprocess_net=net_actor, action_shape=2, softmax_output = False)
critic = DiscreteCritic(preprocess_net=net_critic)

# 4. Policy + Algorithm
ppo_optim = opt.TorchOptimizerFactory(optim_class=torch.optim.Adam, lr=3e-4)

ppo_policy = ProbabilisticActorPolicy(
    actor=actor,
    dist_fn=lambda logits: torch.distributions.Categorical(logits=logits),
    action_space=gym.make("CartPole-v1").action_space,
    action_scaling=False,
)

ppo_algo = PPO(
    policy=ppo_policy,
    critic=critic,
    optim=ppo_optim,
    gamma=0.99,
    gae_lambda=0.95,
    vf_coef=0.5,
    ent_coef=0.01,
    max_grad_norm=0.5,
    eps_clip=0.2,
    value_clip = True,
    recompute_advantage=True,
)

# 5. OptimizerStepMonitor — post-clipping gradient monitoring
from hpo_rl.nets.gradient_monitor import OptimizerStepMonitor

ppo_monitor = OptimizerStepMonitor(
    ppo_algo,
    nets=[net_actor, net_critic],
    log_interval=GRAD_LOG_INTERVAL,
    verbose=GRAD_VERBOSE,
)
print(f"OptimizerStepMonitor attached (log every {GRAD_LOG_INTERVAL} steps)")

# 6. Collectors (on-policy — no persistent buffer)
ppo_train_collector = Collector(ppo_algo, train_envs)
ppo_test_collector  = Collector(ppo_algo, test_envs)

# 7. Train
ppo_params = OnPolicyTrainerParams(
    max_epochs=100,
    epoch_num_steps=2000,
    training_collector=ppo_train_collector,
    test_collector=ppo_test_collector,
    collection_step_num_env_steps=1000,
    update_step_num_repetitions=4,
    test_step_num_episodes=N_TEST_ENVS,
    batch_size=64,
    show_progress=True,
)

t0 = time.perf_counter()
ppo_result = OnPolicyTrainer(algorithm=ppo_algo, params=ppo_params).run()
dt_gm = time.perf_counter() - t0

print(f"\nPPO (GradientMonitoredBaseNet) best reward: {ppo_result.best_reward:.1f}")
print(f"PPO (GradientMonitoredBaseNet) time: {dt_gm:.1f}s")
print(f"PPO actor backward steps (pre-clip):  {net_actor._grad_step}")
print(f"PPO critic backward steps (pre-clip): {net_critic._grad_step}")
print(f"PPO optimizer steps (post-clip):       {ppo_monitor._step}")

# Cleanup
ppo_monitor.remove()
net_actor.remove_hooks()
net_critic.remove_hooks()
train_envs.close()
test_envs.close()

Actor net:  GradientMonitoredBaseNet/actor#1
Critic net: GradientMonitoredBaseNet/critic#2
OptimizerStepMonitor attached (log every 200 steps)
Initial test step: test_reward: 23.750000 ± 9.908961, best_reward: 23.750000 ± 9.908961 in #0


Epoch #1: 100%|##########| 2000/2000 [00:00<00:00, 2447.64it/s, env_episode=81, env_step=2000, len=24, n_ep=36, n_st=1000, rew=28.31, update_step=2]

Epoch #1: test_reward: 32.000000 ± 10.074721, best_reward: 32.000000 ± 10.074721 in #1



Epoch #2: 100%|##########| 2000/2000 [00:00<00:00, 2576.81it/s, env_episode=135, env_step=4000, len=32, n_ep=27, n_st=1000, rew=41.19, update_step=4]

[GradMonitor] GradientMonitoredBaseNet/critic#2 | step 200
  model.0.bias                                       | norm: avg=3.884638 max=11.699697 | val: min=-2.217218 max=0.365810 | mean_abs=0.249021
  model.0.weight                                     | norm: avg=0.564337 max=1.924755 | val: min=-0.252223 max=0.329638 | mean_abs=0.013900
  model.2.bias                                       | norm: avg=2.877223 max=5.766867 | val: min=-1.197766 max=0.652804 | mean_abs=0.152361
  model.2.weight                                     | norm: avg=8.118292 max=17.108259 | val: min=-0.741867 max=0.429183 | mean_abs=0.026675
[GradMonitor] GradientMonitoredBaseNet/actor#1 | step 200
  model.0.bias                                       | norm: avg=0.019327 max=0.064560 | val: min=-0.021939 max=0.017466 | mean_abs=0.001159
  model.0.weight                                     | norm: avg=0.016607 max=0.046702 | val: min=-0.012636 max=0.012708 | mean_abs=0.000403
  model.2.bias                     

...

Epoch #99: 100%|##########| 2000/2000 [00:01<00:00, 1942.82it/s, env_episode=791, env_step=198000, len=61, n_ep=1, n_st=1000, rew=500.00, update_step=198]



Epoch #99: test_reward: 500.000000 ± 0.000000, best_reward: 500.000000 ± 0.000000 in #28


Epoch #100: 100%|##########| 2000/2000 [00:00<00:00, 2277.75it/s, env_episode=797, env_step=200000, len=69, n_ep=4, n_st=1000, rew=500.00, update_step=200]

[GradMonitor] GradientMonitoredBaseNet/critic#2 | step 12000
  model.0.bias                                       | norm: avg=0.630539 max=2.566716 | val: min=-0.435791 max=0.436280 | mean_abs=0.041534
  model.0.weight                                     | norm: avg=0.162574 max=0.627616 | val: min=-0.090718 max=0.079990 | mean_abs=0.003792
  model.2.bias                                       | norm: avg=0.069941 max=0.284884 | val: min=-0.041311 max=0.041377 | mean_abs=0.004622
  model.2.weight                                     | norm: avg=0.342280 max=1.384363 | val: min=-0.033735 max=0.035155 | mean_abs=0.001493
[GradMonitor] GradientMonitoredBaseNet/actor#1 | step 12000
  model.0.bias                                       | norm: avg=0.507773 max=1.795702 | val: min=-0.735980 max=0.675611 | mean_abs=0.026774
  model.0.weight                                     | norm: avg=0.203445 max=0.654404 | val: min=-0.199333 max=0.177008 | mean_abs=0.004418
  model.2.bias                   

Epoch #100: test_reward: 500.000000 ± 0.000000, best_reward: 500.000000 ± 0.000000 in #28

PPO (GradientMonitoredBaseNet) best reward: 500.0
PPO (GradientMonitoredBaseNet) time: 131.2s
PPO actor backward steps (pre-clip):  12000
PPO critic backward steps (pre-clip): 12000
PPO optimizer steps (post-clip):       12000

PPO (GradientMonitoredBaseNet) best reward: 500.0
PPO (GradientMonitoredBaseNet) time: 131.2s
PPO actor backward steps (pre-clip):  12000
PPO critic backward steps (pre-clip): 12000
PPO optimizer steps (post-clip):       12000


## 2b. PPO on CartPole-v1 — standard `Net` (baseline)

Identical hyperparameters, but using tianshou's built-in `Net` instead of `GradientMonitoredBaseNet`.
This lets us compare convergence speed and final reward directly.

In [6]:
# ── PPO on CartPole — standard Net (baseline) ────────────────────
import time

# 1. Environments
train_envs_b = ts.env.DummyVectorEnv([lambda: gym.make("CartPole-v1") for _ in range(N_TRAIN_ENVS)])
test_envs_b  = ts.env.DummyVectorEnv([lambda: gym.make("CartPole-v1") for _ in range(N_TEST_ENVS)])

# 2. Actor + Critic preprocess nets — plain tianshou Net
net_actor_b = Net(state_shape=4, hidden_sizes=[128, 128])
net_critic_b = Net(state_shape=4, hidden_sizes=[128, 128])

print(f"Actor net (Net):  {net_actor_b.model.model}")
print(f"Critic net (Net): {net_critic_b.model.model}")

# 3. Actor + Critic heads (same as GradientMonitoredBaseNet version)
actor_b  = DiscreteActor(preprocess_net=net_actor_b, action_shape=2, softmax_output=False)
critic_b = DiscreteCritic(preprocess_net=net_critic_b)

# 4. Policy + Algorithm — identical hyperparams
ppo_optim_b = opt.TorchOptimizerFactory(optim_class=torch.optim.Adam, lr=3e-4)

ppo_policy_b = ProbabilisticActorPolicy(
    actor=actor_b,
    dist_fn=lambda logits: torch.distributions.Categorical(logits=logits),
    action_space=gym.make("CartPole-v1").action_space,
    action_scaling=False,
)

ppo_algo_b = PPO(
    policy=ppo_policy_b,
    critic=critic_b,
    optim=ppo_optim_b,
    gamma=0.99,
    gae_lambda=0.95,
    vf_coef=0.5,
    ent_coef=0.01,
    max_grad_norm=0.5,
    eps_clip=0.2,
    value_clip=True,
    recompute_advantage=True,
)

# 5. Collectors (on-policy)
ppo_train_collector_b = Collector(ppo_algo_b, train_envs_b)
ppo_test_collector_b  = Collector(ppo_algo_b, test_envs_b)

# 6. Train — same params
ppo_params_b = OnPolicyTrainerParams(
    max_epochs=100,
    epoch_num_steps=2000,
    training_collector=ppo_train_collector_b,
    test_collector=ppo_test_collector_b,
    collection_step_num_env_steps=1000,
    update_step_num_repetitions=4,
    test_step_num_episodes=N_TEST_ENVS,
    batch_size=64,
    show_progress=True,
)

t0 = time.perf_counter()
ppo_result_b = OnPolicyTrainer(algorithm=ppo_algo_b, params=ppo_params_b).run()
dt_net = time.perf_counter() - t0

print(f"\nPPO (Net) best reward: {ppo_result_b.best_reward:.1f}")
print(f"PPO (Net) time: {dt_net:.1f}s")

# Cleanup
train_envs_b.close()
test_envs_b.close()

Actor net (Net):  Sequential(
  (0): Linear(in_features=4, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=128, bias=True)
  (3): ReLU()
)
Critic net (Net): Sequential(
  (0): Linear(in_features=4, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=128, bias=True)
  (3): ReLU()
)
Initial test step: test_reward: 24.500000 ± 12.459936, best_reward: 24.500000 ± 12.459936 in #0


Epoch #1: 100%|##########| 2000/2000 [00:00<00:00, 2741.85it/s, env_episode=75, env_step=2000, len=25, n_ep=33, n_st=1000, rew=27.33, update_step=2]

Epoch #1: test_reward: 23.500000 ± 10.356158, best_reward: 24.500000 ± 12.459936 in #0



Epoch #2: 100%|##########| 2000/2000 [00:00<00:00, 2841.60it/s, env_episode=127, env_step=4000, len=34, n_ep=22, n_st=1000, rew=40.82, update_step=4]

Epoch #2: test_reward: 36.000000 ± 19.506409, best_reward: 36.000000 ± 19.506409 in #2


...

Epoch #98: test_reward: 358.250000 ± 146.352955, best_reward: 500.000000 ± 0.000000 in #32


Epoch #99: 100%|##########| 2000/2000 [00:00<00:00, 2566.04it/s, env_episode=686, env_step=198000, len=42, n_ep=3, n_st=1000, rew=226.00, update_step=198]


Epoch #99: test_reward: 447.000000 ± 91.798693, best_reward: 500.000000 ± 0.000000 in #32


Epoch #100: 100%|##########| 2000/2000 [00:00<00:00, 2837.93it/s, env_episode=688, env_step=200000, len=42, n_ep=1, n_st=1000, rew=500.00, update_step=200]


Epoch #100: test_reward: 500.000000 ± 0.000000, best_reward: 500.000000 ± 0.000000 in #32

PPO (Net) best reward: 500.0
PPO (Net) time: 118.0s


In [8]:
# ── Comparison: GradientMonitoredBaseNet vs Net ──────────────────
print("=" * 60)
print("PPO CartPole-v1 — Convergence Comparison")
print("=" * 60)
print(f"{'Metric':<35} {'GradMonBaseNet':>14} {'Net':>14}")
print("-" * 60)
print(f"{'Best reward':<35} {ppo_result.best_reward:>14.1f} {ppo_result_b.best_reward:>14.1f}")
print(f"{'Best reward std':<35} {ppo_result.best_reward_std:>14.1f} {ppo_result_b.best_reward_std:>14.1f}")
print(f"{'Training time (s)':<35} {dt_gm:>14.1f} {dt_net:>14.1f}")
print("-" * 60)

# Show epoch of first 500.0 from the result attributes
for attr in dir(ppo_result):
    if 'epoch' in attr.lower() or 'best' in attr.lower():
        print(f"  ppo_result.{attr} = {getattr(ppo_result, attr)}")

print()
if ppo_result.best_reward == ppo_result_b.best_reward == 500.0:
    overhead_pct = (dt_gm - dt_net) / dt_net * 100
    print(f"Both reached 500.0")
    print(f"Gradient monitoring overhead: {overhead_pct:+.1f}% ({dt_gm - dt_net:+.1f}s)")
else:
    print("Not both reached 500.0 — check results above")

PPO CartPole-v1 — Convergence Comparison
Metric                              GradMonBaseNet            Net
------------------------------------------------------------
Best reward                                  500.0          500.0
Best reward std                                0.0            0.0
Training time (s)                            131.2          118.0
------------------------------------------------------------
  ppo_result.best_reward = 500.0
  ppo_result.best_reward_std = 0.0
  ppo_result.best_score = 500.0

Both reached 500.0
Gradient monitoring overhead: +11.2% (+13.2s)


---
## 3. Discrete SAC on CartPole-v1

Off-policy actor-critic with **entropy regularization**.

Network instances created:
- **preprocess#1** — actor's preprocess net (no action head)
- **q_net#2** — critic 1 (action_shape=2 → output head for Q-values)
- **q_net#3** — critic 2 (twin critic)

tianshou internally `deepcopy`-s both critics to create target nets.
Our `__deepcopy__` override disables hooks on them → only 3 nets log gradients.

In [6]:
# ── Discrete SAC on CartPole ─────────────────────────────────────
GradientMonitorMixin.reset_instance_counter()

# 1. Environments
train_envs = ts.env.DummyVectorEnv([lambda: gym.make("CartPole-v1") for _ in range(N_TRAIN_ENVS)])
test_envs  = ts.env.DummyVectorEnv([lambda: gym.make("CartPole-v1") for _ in range(N_TEST_ENVS)])

# 2. Actor preprocess net (no action head → DiscreteActor adds softmax head)
sac_net_actor = GradientMonitoredBaseNet(
    state_shape=4,
    hidden_sizes=[128, 128],
    grad_log_interval=GRAD_LOG_INTERVAL,
    grad_verbose=GRAD_VERBOSE,
    grad_monitor_name="actor",
)

# 3. Twin critics — each is a separate GradientMonitoredBaseNet
#    DiscreteSAC gathers Q(s,a) per action → last_size must equal action_shape
sac_net_c1 = GradientMonitoredBaseNet(
    state_shape=4,
    hidden_sizes=[128, 128],
    grad_log_interval=GRAD_LOG_INTERVAL,
    grad_verbose=GRAD_VERBOSE,
    grad_monitor_name="critic1",
)

sac_net_c2 = GradientMonitoredBaseNet(
    state_shape=4,
    hidden_sizes=[128, 128],
    grad_log_interval=GRAD_LOG_INTERVAL,
    grad_verbose=GRAD_VERBOSE,
    grad_monitor_name="critic2",
)

print(f"SAC actor:   {sac_net_actor._gm_name}")
print(f"SAC critic1: {sac_net_c1._gm_name}")
print(f"SAC critic2: {sac_net_c2._gm_name}")

# 4. Heads
#    DiscreteActor: adds softmax output head
#    DiscreteCritic: last_size=2 (one Q-value per action for .gather(1, act))
sac_actor   = DiscreteActor(preprocess_net=sac_net_actor, action_shape=2)
sac_critic1 = DiscreteCritic(preprocess_net=sac_net_c1, last_size=2)
sac_critic2 = DiscreteCritic(preprocess_net=sac_net_c2, last_size=2)

# 5. Policy + Algorithm
sac_optim = opt.TorchOptimizerFactory(optim_class=torch.optim.Adam, lr=3e-4)

sac_policy = DiscreteSACPolicy(
    actor=sac_actor,
    action_space=gym.make("CartPole-v1").action_space,
)

sac_algo = DiscreteSAC(
    policy=sac_policy,
    policy_optim=sac_optim,
    critic=sac_critic1,
    critic_optim=sac_optim,
    critic2=sac_critic2,
    critic2_optim=sac_optim,
    tau=0.005,
    gamma=0.99,
    alpha=AutoAlpha(
        target_entropy=-1.0,
        log_alpha=0.0,
        optim=opt.AdamOptimizerFactory(lr=3e-4),
    ),
)

# 6. Collectors + Buffer
sac_buffer = VectorReplayBuffer(total_size=20000, buffer_num=N_TRAIN_ENVS)
sac_train_collector = Collector(sac_algo, train_envs, sac_buffer)
sac_test_collector  = Collector(sac_algo, test_envs)

# 7. Train
sac_params = OffPolicyTrainerParams(
    max_epochs=10,
    epoch_num_steps=1000,
    training_collector=sac_train_collector,
    test_collector=sac_test_collector,
    collection_step_num_env_steps=200,
    update_step_num_gradient_steps_per_sample=0.5,
    test_step_num_episodes=N_TEST_ENVS,
    batch_size=64,
    show_progress=True,
)

sac_result = OffPolicyTrainer(algorithm=sac_algo, params=sac_params).run()
print(f"\nDiscrete SAC best reward: {sac_result.best_reward:.1f}")
print(f"SAC actor backward steps:   {sac_net_actor._grad_step}")
print(f"SAC critic1 backward steps: {sac_net_c1._grad_step}")
print(f"SAC critic2 backward steps: {sac_net_c2._grad_step}")

# Cleanup
sac_net_actor.remove_hooks()
sac_net_c1.remove_hooks()
sac_net_c2.remove_hooks()
train_envs.close()
test_envs.close()

SAC actor:   GradientMonitoredBaseNet/actor#1
SAC critic1: GradientMonitoredBaseNet/critic1#2
SAC critic2: GradientMonitoredBaseNet/critic2#3
Initial test step: test_reward: 9.000000 ± 0.707107, best_reward: 9.000000 ± 0.707107 in #0


[GradMonitor] GradientMonitoredBaseNet/critic1#2 | step 200
  model.0.bias                                       | norm: avg=0.152991 max=1.749237 | val: min=-0.572933 max=0.521581 | mean_abs=0.008944
  model.0.weight                                     | norm: avg=0.113080 max=1.092914 | val: min=-0.350741 max=0.308221 | mean_abs=0.002592
  model.1.bias                                       | norm: avg=0.058480 max=0.650723 | val: min=-0.203076 max=0.208252 | mean_abs=0.003397
  model.1.weight                                     | norm: avg=0.055058 max=0.664697 | val: min=-0.278095 max=0.209772 | mean_abs=0.002712
  model.3.bias                                       | norm: avg=0.139938 max=1.821341 | val: min=-0.489663 max=0.509988 | mean_abs=0.008308
  model.3.weight                                     | norm: avg=1.042894 max=13.100661 | val: min=-0.729320 max=0.790032 | mean_abs=0.003714
  model.4.bias                                       | norm: avg=0.066267 max=0.733621 | val:

[GradMonitor] GradientMonitoredBaseNet/critic1#2 | step 400
  model.0.bias                                       | norm: avg=0.239505 max=1.039529 | val: min=-0.623215 max=0.480498 | mean_abs=0.013741
  model.0.weight                                     | norm: avg=0.334505 max=1.340409 | val: min=-0.445159 max=0.717985 | mean_abs=0.007464
  model.1.bias                                       | norm: avg=0.113433 max=0.467807 | val: min=-0.267309 max=0.183081 | mean_abs=0.006469
  model.1.weight                                     | norm: avg=0.076092 max=0.257364 | val: min=-0.093863 max=0.101470 | mean_abs=0.004311
  model.3.bias                                       | norm: avg=0.133875 max=0.389501 | val: min=-0.115750 max=0.112213 | mean_abs=0.008644
  model.3.weight                                     | norm: avg=1.037714 max=3.128551 | val: min=-0.260385 max=0.181398 | mean_abs=0.004358
  model.4.bias                                       | norm: avg=0.054430 max=0.139722 | val: 

Epoch #1: 100%|##########| 1000/1000 [00:04<00:00, 224.28it/s, env_episode=40, env_step=1000, len=19, n_ep=6, n_st=200, rew=19.83, update_step=5]

Epoch #1: test_reward: 21.250000 ± 3.491060, best_reward: 21.250000 ± 3.491060 in #1


...

Epoch #9: test_reward: 472.000000 ± 45.656325, best_reward: 472.000000 ± 45.656325 in #9


[GradMonitor] GradientMonitoredBaseNet/critic1#2 | step 4600
  model.0.bias                                       | norm: avg=7.751651 max=33.975594 | val: min=-21.244049 max=10.993559 | mean_abs=0.392798
  model.0.weight                                     | norm: avg=7.280958 max=27.709402 | val: min=-7.920628 max=11.492371 | mean_abs=0.159458
  model.1.bias                                       | norm: avg=2.670859 max=10.149425 | val: min=-5.649541 max=3.470030 | mean_abs=0.140471
  model.1.weight                                     | norm: avg=1.501403 max=5.495087 | val: min=-3.312226 max=1.725620 | mean_abs=0.083105
  model.3.bias                                       | norm: avg=1.649865 max=6.341030 | val: min=-0.956174 max=1.457675 | mean_abs=0.109359
  model.3.weight                                     | norm: avg=11.877564 max=46.094135 | val: min=-1.970601 max=2.774689 | mean_abs=0.050301
  model.4.bias                                       | norm: avg=0.393260 max=1.25067

[GradMonitor] GradientMonitoredBaseNet/critic1#2 | step 4800
  model.0.bias                                       | norm: avg=9.998704 max=64.558563 | val: min=-31.248634 max=37.368088 | mean_abs=0.508251
  model.0.weight                                     | norm: avg=9.071601 max=55.980289 | val: min=-15.280038 max=26.066343 | mean_abs=0.198733
  model.1.bias                                       | norm: avg=3.431328 max=20.809973 | val: min=-8.387166 max=11.190328 | mean_abs=0.181153
  model.1.weight                                     | norm: avg=1.928084 max=11.582062 | val: min=-4.166111 max=4.919449 | mean_abs=0.106187
  model.3.bias                                       | norm: avg=2.123568 max=12.809675 | val: min=-3.020446 max=2.505471 | mean_abs=0.139883
  model.3.weight                                     | norm: avg=14.912215 max=85.700996 | val: min=-6.553410 max=4.787213 | mean_abs=0.062209
  model.4.bias                                       | norm: avg=0.537248 max=2.5

Epoch #10: 100%|##########| 1000/1000 [00:04<00:00, 226.10it/s, env_episode=160, env_step=10000, len=149, n_ep=2, n_st=200, rew=149.00, update_step=50]


[GradMonitor] GradientMonitoredBaseNet/critic1#2 | step 5000
  model.0.bias                                       | norm: avg=10.859334 max=53.452621 | val: min=-31.748093 max=21.019270 | mean_abs=0.528881
  model.0.weight                                     | norm: avg=9.823440 max=32.410217 | val: min=-12.181371 max=14.961300 | mean_abs=0.208752
  model.1.bias                                       | norm: avg=3.669887 max=14.221720 | val: min=-7.892976 max=6.462842 | mean_abs=0.186837
  model.1.weight                                     | norm: avg=2.084218 max=8.187583 | val: min=-5.729163 max=4.081971 | mean_abs=0.109392
  model.3.bias                                       | norm: avg=2.152006 max=7.947305 | val: min=-1.871813 max=1.582815 | mean_abs=0.143258
  model.3.weight                                     | norm: avg=15.316717 max=60.847778 | val: min=-3.453150 max=3.625662 | mean_abs=0.063740
  model.4.bias                                       | norm: avg=0.521756 max=1.865

## Summary

| Algorithm | Net class | Monitoring | Notes |
|-----------|-----------|------------|-------|
| **DQN** | `GradientMonitoredNet` | pre-clip hooks | Single Q-net, stable |
| **PPO** | `GradientMonitoredBaseNet` | pre-clip hooks + `OptimizerStepMonitor` | Actor+Critic, shows pre vs post clipping |
| **Discrete SAC** | `GradientMonitoredBaseNet` | pre-clip hooks | Actor + 2 critics |

### Key observations
- **DQN**: Gradients are small and stable on CartPole.
- **PPO**: Critic gradients **before** `clip_grad_norm_` reach norm ~100+ (`[EXPLODING]`), but **after** clipping they stay under `max_grad_norm=0.5`. This is normal behavior — the large pre-clip norms come from the value function's MSE loss scale.
- **SAC**: Separate optimizers per net — gradients are healthy.

### `OptimizerStepMonitor` usage
```python
from hpo_rl.nets.gradient_monitor import OptimizerStepMonitor

monitor = OptimizerStepMonitor(algorithm, nets=[net1, net2], log_interval=200)
# ... train ...
monitor.remove()  # cleanup
```

In [1]:
"""
Pendulum-v1 continuous action space test:
  PPO  + GradientMonitoredBaseNet  vs  PPO  + Net
  SAC  + GradientMonitoredBaseNet  vs  SAC  + Net
"""

import time
import gymnasium as gym
import torch
import numpy as np
import sys


from torch.distributions import Independent, Normal

from tianshou.algorithm.modelfree.ppo import PPO
from tianshou.algorithm.modelfree.reinforce import ProbabilisticActorPolicy
from tianshou.algorithm.modelfree.sac import SAC, SACPolicy, AutoAlpha
from tianshou.algorithm.optim import TorchOptimizerFactory, AdamOptimizerFactory
from tianshou.utils.net.continuous import ContinuousActorProbabilistic, ContinuousCritic
from tianshou.utils.net.common import Net
from tianshou.data import VectorReplayBuffer, Collector
from tianshou.env import DummyVectorEnv
from tianshou.trainer import (
    OffPolicyTrainer, OffPolicyTrainerParams,
    OnPolicyTrainer, OnPolicyTrainerParams,
)

from hpo_rl.nets.gradient_monitor import (
    GradientMonitoredBaseNet,
    GradientMonitorMixin,
    OptimizerStepMonitor,
)



In [2]:

# ────────────────────────────────────────────────────────────────────
#  Constants
# ────────────────────────────────────────────────────────────────────
ENV_NAME = "Pendulum-v1"
OBS_DIM = 3
ACT_DIM = 1
HIDDEN = (64, 64)
N_TRAIN = 8
N_TEST = 4
GRAD_LOG_INTERVAL = 500
GRAD_VERBOSE = True

results = {}


# ════════════════════════════════════════════════════════════════════
def make_envs():
    train = DummyVectorEnv([lambda: gym.make(ENV_NAME) for _ in range(N_TRAIN)])
    test  = DummyVectorEnv([lambda: gym.make(ENV_NAME) for _ in range(N_TEST)])
    return train, test

In [11]:

# ════════════════════════════════════════════════════════════════════
#  1.  PPO + GradientMonitoredBaseNet
# ════════════════════════════════════════════════════════════════════
def run_ppo_gradmon():
    print("\n" + "="*70)
    print("  PPO  +  GradientMonitoredBaseNet   (Pendulum-v1)")
    print("="*70)
    GradientMonitorMixin.reset_instance_counter()
    train_envs, test_envs = make_envs()

    net_actor = GradientMonitoredBaseNet(
        state_shape=OBS_DIM, hidden_sizes=HIDDEN,
        grad_log_interval=GRAD_LOG_INTERVAL, grad_verbose=GRAD_VERBOSE,
        grad_monitor_name="ppo_actor",
    )
    actor = ContinuousActorProbabilistic(
        preprocess_net=net_actor,
        action_shape=ACT_DIM,
        hidden_sizes=(),
        max_action=1.0,
        unbounded=True,
        conditioned_sigma=True,
    )

    net_critic = GradientMonitoredBaseNet(
        state_shape=OBS_DIM, hidden_sizes=HIDDEN,
        grad_log_interval=GRAD_LOG_INTERVAL, grad_verbose=GRAD_VERBOSE,
        grad_monitor_name="ppo_critic",
    )
    critic = ContinuousCritic(preprocess_net=net_critic)

    env_for_space = gym.make(ENV_NAME)
    policy = ProbabilisticActorPolicy(
        actor=actor,
        dist_fn=lambda mu_sigma: Independent(Normal(*mu_sigma), 1),
        action_space=env_for_space.action_space,
        action_scaling=True,
        action_bound_method="clip",
    )
    env_for_space.close()

    optim = TorchOptimizerFactory(torch.optim.Adam, lr=3e-4)

    algo = PPO(
        policy=policy,
        critic=critic,
        optim=optim,
        gamma=0.99,
        gae_lambda=0.95,
        eps_clip=0.2,
        vf_coef=0.5,
        ent_coef=0.01,
        max_grad_norm=0.5,
        recompute_advantage=True,
    )

    monitor = OptimizerStepMonitor(
        algo, nets=[net_actor, net_critic],
        log_interval=GRAD_LOG_INTERVAL, verbose=GRAD_VERBOSE,
    )

    train_col = Collector(algo, train_envs)
    test_col  = Collector(algo, test_envs)

    params = OnPolicyTrainerParams(
        max_epochs=50,
        epoch_num_steps=2000,
        training_collector=train_col,
        test_collector=test_col,
        collection_step_num_env_steps=1000,
        update_step_num_repetitions=10,
        test_step_num_episodes=N_TEST,
        batch_size=64,
        show_progress=True,
    )

    t0 = time.perf_counter()
    result = OnPolicyTrainer(algorithm=algo, params=params).run()
    dt = time.perf_counter() - t0

    best = result.best_reward
    print(f"\n>>> PPO+GradMon  best_reward={best:.1f}  time={dt:.1f}s")
    print(f"    actor grad steps={net_actor._grad_step}  critic grad steps={net_critic._grad_step}")
    print(f"    optimizer monitor steps={monitor._step}")

    monitor.remove()
    net_actor.remove_hooks()
    net_critic.remove_hooks()
    train_envs.close(); test_envs.close()
    return best, dt


In [12]:

# ════════════════════════════════════════════════════════════════════
#  2.  PPO + Net (baseline)
# ════════════════════════════════════════════════════════════════════
def run_ppo_net():
    print("\n" + "="*70)
    print("  PPO  +  Net  (baseline)  (Pendulum-v1)")
    print("="*70)
    train_envs, test_envs = make_envs()

    net_actor = Net(state_shape=OBS_DIM, hidden_sizes=HIDDEN)
    actor = ContinuousActorProbabilistic(
        preprocess_net=net_actor,
        action_shape=ACT_DIM,
        hidden_sizes=(),
        max_action=1.0,
        unbounded=True,
        conditioned_sigma=True,
    )

    net_critic = Net(state_shape=OBS_DIM, hidden_sizes=HIDDEN)
    critic = ContinuousCritic(preprocess_net=net_critic)

    env_for_space = gym.make(ENV_NAME)
    policy = ProbabilisticActorPolicy(
        actor=actor,
        dist_fn=lambda mu_sigma: Independent(Normal(*mu_sigma), 1),
        action_space=env_for_space.action_space,
        action_scaling=True,
        action_bound_method="clip",
    )
    env_for_space.close()

    optim = TorchOptimizerFactory(torch.optim.Adam, lr=3e-4)

    algo = PPO(
        policy=policy,
        critic=critic,
        optim=optim,
        gamma=0.99,
        gae_lambda=0.95,
        eps_clip=0.2,
        vf_coef=0.5,
        ent_coef=0.01,
        max_grad_norm=0.5,
        recompute_advantage=True,
    )

    train_col = Collector(algo, train_envs)
    test_col  = Collector(algo, test_envs)

    params = OnPolicyTrainerParams(
        max_epochs=50,
        epoch_num_steps=2000,
        training_collector=train_col,
        test_collector=test_col,
        collection_step_num_env_steps=1000,
        update_step_num_repetitions=10,
        test_step_num_episodes=N_TEST,
        batch_size=64,
        show_progress=True,
    )

    t0 = time.perf_counter()
    result = OnPolicyTrainer(algorithm=algo, params=params).run()
    dt = time.perf_counter() - t0

    best = result.best_reward
    print(f"\n>>> PPO+Net  best_reward={best:.1f}  time={dt:.1f}s")
    train_envs.close(); test_envs.close()
    return best, dt


In [3]:

# ════════════════════════════════════════════════════════════════════
#  3.  SAC + GradientMonitoredBaseNet
# ════════════════════════════════════════════════════════════════════
def run_sac_gradmon():
    print("\n" + "="*70)
    print("  SAC  +  GradientMonitoredBaseNet   (Pendulum-v1)")
    print("="*70)
    GradientMonitorMixin.reset_instance_counter()
    train_envs, test_envs = make_envs()

    # --- actor ---
    net_actor = GradientMonitoredBaseNet(
        state_shape=OBS_DIM, hidden_sizes=HIDDEN,
        grad_log_interval=GRAD_LOG_INTERVAL, grad_verbose=GRAD_VERBOSE,
        grad_monitor_name="sac_actor",
    )
    actor = ContinuousActorProbabilistic(
        preprocess_net=net_actor,
        action_shape=ACT_DIM,
        hidden_sizes=(),
        max_action=1.0,
        unbounded=True,
        conditioned_sigma=True,
    )

    env_for_space = gym.make(ENV_NAME)
    sac_policy = SACPolicy(
        actor=actor,
        action_space=env_for_space.action_space,
        action_scaling=True,
    )
    env_for_space.close()

    # --- critics (Q(s,a)) ---
    # ContinuousCritic(apply_preprocess_net_to_obs_only=False) concatenates [obs, act]
    # before passing to preprocess_net => preprocess_net input_dim = obs+act
    net_c1 = GradientMonitoredBaseNet(
        state_shape=OBS_DIM, action_shape=ACT_DIM, concat=True,
        hidden_sizes=HIDDEN,
        grad_log_interval=GRAD_LOG_INTERVAL, grad_verbose=GRAD_VERBOSE,
        grad_monitor_name="sac_c1",
    )
    critic1 = ContinuousCritic(preprocess_net=net_c1)

    net_c2 = GradientMonitoredBaseNet(
        state_shape=OBS_DIM, action_shape=ACT_DIM, concat=True,
        hidden_sizes=HIDDEN,
        grad_log_interval=GRAD_LOG_INTERVAL, grad_verbose=GRAD_VERBOSE,
        grad_monitor_name="sac_c2",
    )
    critic2 = ContinuousCritic(preprocess_net=net_c2)

    policy_optim  = TorchOptimizerFactory(torch.optim.Adam, lr=3e-4)
    critic_optim  = TorchOptimizerFactory(torch.optim.Adam, lr=3e-4)
    critic2_optim = TorchOptimizerFactory(torch.optim.Adam, lr=3e-4)

    algo = SAC(
        policy=sac_policy,
        policy_optim=policy_optim,
        critic=critic1,
        critic_optim=critic_optim,
        critic2=critic2,
        critic2_optim=critic2_optim,
        tau=0.005,
        gamma=0.99,
        alpha=AutoAlpha(
            target_entropy=-float(ACT_DIM),
            log_alpha=0.0,
            optim=AdamOptimizerFactory(lr=3e-4),
        ),
    )

    buf = VectorReplayBuffer(total_size=100000, buffer_num=N_TRAIN)
    train_col = Collector(algo, train_envs, buf)
    test_col  = Collector(algo, test_envs)

    # warm-up with random actions
    train_col.collect(n_step=2000, random=True, reset_before_collect=True)

    params = OffPolicyTrainerParams(
        max_epochs=30,
        epoch_num_steps=4000,
        training_collector=train_col,
        test_collector=test_col,
        collection_step_num_env_steps=10,
        update_step_num_gradient_steps_per_sample=0.1,
        test_step_num_episodes=N_TEST,
        batch_size=256,
        show_progress=True,
    )

    t0 = time.perf_counter()
    result = OffPolicyTrainer(algorithm=algo, params=params).run()
    dt = time.perf_counter() - t0

    best = result.best_reward
    print(f"\n>>> SAC+GradMon  best_reward={best:.1f}  time={dt:.1f}s")
    print(f"    actor grad steps={net_actor._grad_step}")
    print(f"    critic1 grad steps={net_c1._grad_step}")
    print(f"    critic2 grad steps={net_c2._grad_step}")

    net_actor.remove_hooks()
    net_c1.remove_hooks()
    net_c2.remove_hooks()
    train_envs.close(); test_envs.close()
    return best, dt

In [4]:
run_sac_gradmon()


  SAC  +  GradientMonitoredBaseNet   (Pendulum-v1)
Initial test step: test_reward: -1117.446993 ± 112.726329, best_reward: -1117.446993 ± 112.726329 in #0
Initial test step: test_reward: -1117.446993 ± 112.726329, best_reward: -1117.446993 ± 112.726329 in #0


Epoch #1:   0%|          | 0/4000 [00:00<?, ?it/s]c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\tianshou\data\collector.py:539: UserWarning: n_step=10 is not a multiple of (self.env_num=8), which may cause extra transitions being collected into the buffer.
  warnings.warn(
c:\Users\Administrator\AppData\Local\Programs\Python\Python313\Lib\site-packages\tianshou\data\collector.py:539: UserWarning: n_step=10 is not a multiple of (self.env_num=8), which may cause extra transitions being collected into the buffer.
  warnings.warn(


[GradMonitor] GradientMonitoredBaseNet/sac_c2#3 | step 500
  model.0.bias                                       | norm: avg=2.266495 max=3.807359 | val: min=-1.313468 max=0.381900 | mean_abs=0.231583
  model.0.weight                                     | norm: avg=5.163792 max=9.931621 | val: min=-1.778705 max=2.502536 | mean_abs=0.193966
  model.2.bias                                       | norm: avg=2.537373 max=4.853427 | val: min=-1.405008 max=1.068699 | mean_abs=0.219352
  model.2.weight                                     | norm: avg=10.419393 max=20.316357 | val: min=-1.590523 max=1.096649 | mean_abs=0.093268
[GradMonitor] GradientMonitoredBaseNet/sac_c1#2 | step 500
  model.0.bias                                       | norm: avg=1.896881 max=3.540686 | val: min=-1.169621 max=0.281011 | mean_abs=0.195959
  model.0.weight                                     | norm: avg=4.530299 max=8.888237 | val: min=-2.113441 max=1.918185 | mean_abs=0.167911
  model.2.bias                    

Epoch #1: 100%|##########| 4000/4000 [00:05<00:00, 680.97it/s, env_episode=16, env_step=4000, n_ep=0, n_st=16, update_step=250]


[GradMonitor] GradientMonitoredBaseNet/sac_c2#3 | step 1000
  model.0.bias                                       | norm: avg=1.749684 max=3.363522 | val: min=-1.171017 max=0.260199 | mean_abs=0.170844
  model.0.weight                                     | norm: avg=4.143427 max=6.767527 | val: min=-1.748677 max=1.996900 | mean_abs=0.164703
  model.2.bias                                       | norm: avg=0.563909 max=1.386402 | val: min=-0.345251 max=0.112065 | mean_abs=0.050823
  model.2.weight                                     | norm: avg=3.284950 max=5.547629 | val: min=-0.402717 max=0.431219 | mean_abs=0.031149
[GradMonitor] GradientMonitoredBaseNet/sac_c1#2 | step 1000
  model.0.bias                                       | norm: avg=1.196655 max=2.624974 | val: min=-0.821789 max=0.506187 | mean_abs=0.119324
  model.0.weight                                     | norm: avg=3.113989 max=5.396546 | val: min=-1.715087 max=1.502801 | mean_abs=0.120803
  model.2.bias                    

...

Epoch #30:   0%|          | 0/4000 [00:00<?, ?it/s]

[GradMonitor] GradientMonitoredBaseNet/sac_c2#3 | step 29500
  model.0.bias                                       | norm: avg=54.540905 max=162.328629 | val: min=-55.523434 max=77.782013 | mean_abs=4.930671 [EXPLODING]
  model.0.weight                                     | norm: avg=208.438862 max=699.813293 | val: min=-295.399933 max=274.837891 | mean_abs=6.791637 [EXPLODING]
  model.2.bias                                       | norm: avg=4.699168 max=17.210913 | val: min=-5.595210 max=3.590240 | mean_abs=0.485052
  model.2.weight                                     | norm: avg=47.409083 max=131.880676 | val: min=-23.509649 max=17.035646 | mean_abs=0.494046 [EXPLODING]
[GradMonitor] GradientMonitoredBaseNet/sac_c1#2 | step 29500
  model.0.bias                                       | norm: avg=75.136021 max=215.550446 | val: min=-98.843262 max=120.138275 | mean_abs=6.596564 [EXPLODING]
  model.0.weight                                     | norm: avg=284.557174 max=863.698730 | val: mi

Epoch #30: 100%|##########| 4000/4000 [00:07<00:00, 562.75it/s, env_episode=600, env_step=120000, len=200, n_ep=8, n_st=16, rew=-165.38, update_step=7500]


[GradMonitor] GradientMonitoredBaseNet/sac_c2#3 | step 30000
  model.0.bias                                       | norm: avg=51.399415 max=150.437378 | val: min=-52.327263 max=68.540016 | mean_abs=4.659166 [EXPLODING]
  model.0.weight                                     | norm: avg=200.611663 max=582.516357 | val: min=-220.527512 max=238.741165 | mean_abs=6.568510 [EXPLODING]
  model.2.bias                                       | norm: avg=4.339267 max=16.150948 | val: min=-5.234533 max=3.289918 | mean_abs=0.442990
  model.2.weight                                     | norm: avg=44.625317 max=111.948380 | val: min=-18.835697 max=17.546419 | mean_abs=0.462087 [EXPLODING]
[GradMonitor] GradientMonitoredBaseNet/sac_c1#2 | step 30000
  model.0.bias                                       | norm: avg=68.307889 max=183.467850 | val: min=-76.322159 max=118.248749 | mean_abs=6.061444 [EXPLODING]
  model.0.weight                                     | norm: avg=287.050158 max=783.090210 | val: mi

(-31.635556876123996, 206.91519849997712)

In [14]:

# ════════════════════════════════════════════════════════════════════
#  4.  SAC + Net (baseline)
# ════════════════════════════════════════════════════════════════════
def run_sac_net():
    print("\n" + "="*70)
    print("  SAC  +  Net  (baseline)  (Pendulum-v1)")
    print("="*70)
    train_envs, test_envs = make_envs()

    net_actor = Net(state_shape=OBS_DIM, hidden_sizes=HIDDEN)
    actor = ContinuousActorProbabilistic(
        preprocess_net=net_actor,
        action_shape=ACT_DIM,
        hidden_sizes=(),
        max_action=1.0,
        unbounded=True,
        conditioned_sigma=True,
    )

    env_for_space = gym.make(ENV_NAME)
    sac_policy = SACPolicy(
        actor=actor,
        action_space=env_for_space.action_space,
        action_scaling=True,
    )
    env_for_space.close()

    net_c1 = Net(state_shape=OBS_DIM, action_shape=ACT_DIM, hidden_sizes=HIDDEN, concat=True)
    critic1 = ContinuousCritic(preprocess_net=net_c1)

    net_c2 = Net(state_shape=OBS_DIM, action_shape=ACT_DIM, hidden_sizes=HIDDEN, concat=True)
    critic2 = ContinuousCritic(preprocess_net=net_c2)

    policy_optim  = TorchOptimizerFactory(torch.optim.Adam, lr=3e-4)
    critic_optim  = TorchOptimizerFactory(torch.optim.Adam, lr=3e-4)
    critic2_optim = TorchOptimizerFactory(torch.optim.Adam, lr=3e-4)

    algo = SAC(
        policy=sac_policy,
        policy_optim=policy_optim,
        critic=critic1,
        critic_optim=critic_optim,
        critic2=critic2,
        critic2_optim=critic2_optim,
        tau=0.005,
        gamma=0.99,
        alpha=AutoAlpha(
            target_entropy=-float(ACT_DIM),
            log_alpha=0.0,
            optim=AdamOptimizerFactory(lr=3e-4),
        ),
    )

    buf = VectorReplayBuffer(total_size=100000, buffer_num=N_TRAIN)
    train_col = Collector(algo, train_envs, buf)
    test_col  = Collector(algo, test_envs)

    train_col.collect(n_step=2000, random=True, reset_before_collect=True)

    params = OffPolicyTrainerParams(
        max_epochs=30,
        epoch_num_steps=4000,
        training_collector=train_col,
        test_collector=test_col,
        collection_step_num_env_steps=10,
        update_step_num_gradient_steps_per_sample=0.1,
        test_step_num_episodes=N_TEST,
        batch_size=256,
        show_progress=True,
    )

    t0 = time.perf_counter()
    result = OffPolicyTrainer(algorithm=algo, params=params).run()
    dt = time.perf_counter() - t0

    best = result.best_reward
    print(f"\n>>> SAC+Net  best_reward={best:.1f}  time={dt:.1f}s")
    train_envs.close(); test_envs.close()
    return best, dt
